<a href="https://colab.research.google.com/github/roxo659/projet-TPOO-/blob/main/diabetes_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Purpose of this notebook
This notebook aims to build a model that determines whether a person is prone to diabetes or not. Additionally, it seeks to identify a subset of features (risk factors) that can accurately predict the risk of diabetes. The weights of the optimal solution will be utilized in another project, where they will be applied to users' inputs in real time.

## Dataset
This notebook makes use of a subset of a larger dataset which aimed to collect uniform, state-specific data on preventive health practices and risk behaviors that are associated with chronic diseases, injuries, and preventable infectious diseases in the adult population. The subset used in this notebook can be accessed [here](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset?select=diabetes_binary_5050split_health_indicators_BRFSS2015.csv).

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense,LeakyReLU,Dropout
from keras.optimizers import Adagrad, RMSprop, Adam
from keras.regularizers import l1, l2
from keras.initializers import he_normal
from keras.activations import selu
from keras.metrics import Precision, Recall
from sklearn.decomposition import PCA

In [2]:
drive.mount('/drive')
DATASET_ADDRESS = '/drive/MyDrive/diabetes_info.csv'
raw_dataset = pd.read_csv(DATASET_ADDRESS)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
raw_dataset.info()

In [ ]:
raw_dataset = raw_dataset.drop(columns=['Income']) # Income unit used in this research is USD which is not scalable to Rials. So we ignore it.

In [ ]:
y = raw_dataset["Diabetes_binary"]
x = raw_dataset.drop(columns=["Diabetes_binary"])

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=9)

# Model selection
While our data may appear relatively clean, this does not guarantee optimal performance. Therefore, we must leverage a range of machine learning models to assess their effectiveness and identify potential modifications to the original data that can enhance the performance of our models.

## First model: Gradient boost classifier
Boosting algorithms have been widely recognized as effective choices for handling tabular data. Among them, gradient boosting stands out as a prominent technique that leverages decision trees to create a powerful ensemble model. Nonetheless, to ensure its optimal performance, careful consideration should be given to hyperparameter tuning.

In [ ]:
def get_data(dataset):
  y = dataset["Diabetes_binary"]
  x = dataset.drop(columns=["Diabetes_binary"])
  return train_test_split(x, y, test_size=0.20, random_state=9)

In [ ]:
def gradient_boost_classifier_model(dataset, learning_rate=0.05, n_estimators=150, subsample=0.8):
  x_train, x_test, y_train, y_test = get_data(dataset)
  reg = GradientBoostingClassifier(random_state=90,
                                loss='deviance',
                                learning_rate=learning_rate,
                                n_estimators=n_estimators,
                                subsample=subsample,
                                criterion='friedman_mse',
                                verbose=2,
                                )
  reg.fit(x_train, y_train)
  y_pred = reg.predict(x_test)
  report = classification_report(y_test, y_pred)
  print(report)

In [ ]:
gradient_boost_classifier_model(raw_dataset)

## The deviance loss
Deviance loss is a commonly used loss function in binary classification problems. With a glance at its formula, we can easily unserstand why:

$$
L(y, p) = \left(y \log(p) + (1 - y) \log(1 - p)\right)
$$

where y is true class and p is statistical probability.




## F-1 score
F-1 score uses precision(ratio of true possitives to true possitves and false possitives) and recall(ratio of true possitives to true possitves and false negatives) scores to prvoide a balance between them:

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$


In [ ]:
# scores = cross_val_score(reg, x_train, y_train, cv=5, scoring='f1_macro')
# print("cross validation scores(F-1) where k=5: ", scores)

In [ ]:
# scores = cross_val_score(reg, x_train, y_train, cv=10, scoring='f1_macro')
# print("cross validation scores(F-1) where k=10: ", scores)

## Initial evaluataion result
As demonstrated above, whether employing Gradient Boosting with or without cross-validation, the F1 score hovers around 0.75. While this performance is acceptable, there is room for improvement.

# Second model: Logistic regression
While Logistic Regression is typically considered a more linear model compared to ensemble methods, it remains a highly prevalent choice in classification problems. It offers several distinct advantages, such as strong interpretability, feature importance insights, and the ability to not only make binary classifications but also provide class probabilities. This probabilistic aspect can prove particularly valuable in certain situations."







In [ ]:
def logistic_regression_model(dataset):
  x_train, x_test, y_train, y_test = get_data(dataset)
  log_reg = LogisticRegression(random_state=32, solver='sag', multi_class='multinomial', verbose=2, max_iter=500)
  log_reg.fit(x_train, y_train)
  y_pred_log_reg = log_reg.predict(x_test)
  report_log_reg = classification_report(y_test, y_pred_log_reg)
  print(log_reg.coef_)
  print(report_log_reg)

In [ ]:
logistic_regression_model(raw_dataset)

## Evaluation result
Logistic regression exhibited slightly lower performance compared to Gradient Boosting, indicating that additional data preprocessing may be necessary to enhance model outcomes.

### Checking for class imbalance

In [ ]:
np.bincount(y)

## Standardizing features
In this section we standardize featuers that their domain may mislead oue models.

In [ ]:
columns_to_standardize = list(x.keys())

In [ ]:
scaler = StandardScaler()
standarized_features = scaler.fit_transform(raw_dataset[columns_to_standardize])
standardized_dataset = pd.DataFrame()
standardized_dataset["Diabetes_binary"] = raw_dataset["Diabetes_binary"]
standardized_dataset[columns_to_standardize] = standarized_features

In [ ]:
gradient_boost_classifier_model(standardized_dataset)

In [ ]:
logistic_regression_model(standardized_dataset)

## Normalizing features
Standardization helped the convergance of our model, but didn't countribute to the evaluation metrics. Now we try with normalized data.

In [ ]:
columns_to_normalize = list(x.keys())

In [ ]:
min_max_scaler = MinMaxScaler()
normalized_features = min_max_scaler.fit_transform(raw_dataset[columns_to_standardize])
normalized_dataset = pd.DataFrame()
normalized_dataset["Diabetes_binary"] = raw_dataset["Diabetes_binary"]
normalized_dataset[columns_to_standardize] = normalized_features

In [ ]:
gradient_boost_classifier_model(normalized_dataset)

In [ ]:
logistic_regression_model(normalized_dataset)

# What happened?
It turned out that algorithms like Logitstic regression and Gradientboost are robust to data scale due to a number of factors like their loss functions, use of decision trees and regularization factors, etc. So we have to find another way to reach our goal.

# Next model: DNN
neural networks are the master of finding complex relations between featurse. In addition to that, they can be combined with various functionalities to improve model's behavoir even further, e.g. optimizers, regularization factors, etc.

In [ ]:
def dnn_model(dataset):
  x_train, x_test, y_train, y_test = get_data(dataset)
  model = Sequential()
  model.add(Dense(64, input_dim=x_train.shape[1], activation=LeakyReLU(alpha=0.1), kernel_initializer=he_normal()))
  model.add(Dropout(0.5))
  model.add(Dense(128, activation='relu'))
  model.add(Dense(32, activation='relu', kernel_regularizer=l1(0.1)))
  model.add(Dense(1, activation='sigmoid'))
  adam = Adagrad(learning_rate=0.1)
  model.compile(loss='binary_crossentropy', optimizer=adam, metrics=[Precision(), Recall()])
  model.fit(x_train, y_train, epochs=100, verbose=2, validation_split=0.1, batch_size=100,)
  res = model.evaluate(x_test, y_test)
  print("binary cross-entropy loss : ", res[0], " precision: ", res[1], " recal: ", res[2])

In [ ]:
dnn_model(standardized_dataset)

## Result
As we saw, different models are not showing significant result improvements. So we may need to make some changes to our data

## The correlation matrix and its usage
Correlation matrix simply explains the relationship between columns of a dataset. The correlation coefficient ranges between -1 and 1. A correlation coefficient of 1 indicates a perfect positive correlation, meaning that the two variables increase or decrease together in a linear relationship. A correlation coefficient of -1 indicates a perfect negative correlation, meaning that the two variables move in opposite directions in a linear relationship. A correlation coefficient close to 0 suggests no linear relationship between the variables.

This matrix can be helpful when finding an optimal subset of features.

In [ ]:
def sort_correlations(correlation_matrix):
  return sorted(correlation_matrix.items(), key=lambda x:abs(x[1]))

In [ ]:
def get_correlations(dataset):
  columns = dataset.keys()
  correlation = dataset[columns].corr()
  return correlation["Diabetes_binary"]

In [ ]:
correlation_map = sort_correlations(get_correlations(raw_dataset))

In [ ]:
correlation_map

In [ ]:
keep_features = map(lambda b: b[0], filter(lambda a: abs(a[1]) > 0.25, correlation_map))

In [ ]:
modified_dataset = raw_dataset[list(keep_features)]

In [ ]:
logistic_regression_model(modified_dataset)

# Feature selection result: Logistic regression
By reducing the feature count using the correlation matrix and only keeping faetures that have more meaningful relationship with the target featurse, Logistic regression not only converged faster, it also kept its accuracy.

In [ ]:
gradient_boost_classifier_model(modified_dataset)

# Feature selection result: GradientBoost
GradientBoost was also capable of keeping its performance after feature selection. It is worthy of noting that tuning hyperparameters had a mild effect on this model but it was negligible.

In [ ]:
dnn_model(modified_dataset)

# Feature selection result: DNN
DNN also proved to be consistant after feature selection. It even had a mild improvement(which is again, negligible).

# Feature selection overall result
So to conclude, we were able to predict our target feature with an acceptable accuracy even after selecting a subsest of our features. The following is the list of remaining features which proved to be decisive: DiffWalk, Age, HighChol, BMI, HighBP, GenHlth


In [ ]:
# rename functions to snake_case

In [ ]:
def get_feature_name(count):
  return ["f{c}".format(c=i) for i in range(count)]

In [ ]:
component_count = 15

all_features = list(raw_dataset.keys())

all_features.remove("Diabetes_binary")

pca = PCA(n_components = component_count)

pca_columns = pca.fit_transform(standardized_dataset[all_features])

In [ ]:
pca_dataset = pd.DataFrame()
pca_dataset["Diabetes_binary"] = standardized_dataset["Diabetes_binary"]
pca_dataset[get_feature_name(component_count)] =  pca_columns

In [ ]:
logistic_regression_model(pca_dataset)

In [ ]:
gradient_boost_classifier_model(pca_dataset)

In [ ]:
dnn_model(pca_dataset)

# Feature reduction: PCA
Principle component analysis is a technique which is used to decrease feature count such that their information is preserved, only in lower dimensions. It mainly aims to keep the variance of the original data in fewer columns. Result however, did not show any significant improvement, suggesting that important relations between features and target are already captured.  



# Final result
for the purpose of our research, we will use the following weights obtained from the Logistic regression model to apply on users' input.

### feature selected model:
$$[0.0702186,  0.08500237, 0.29923042, 0.03782045, 0.39012718, 0.30067847]$$


### original model:
$$[ 0.37257845,  0.28995716,  0.66909954,  0.0368971,  -0.00249678,  0.09599408,
   0.11823623, -0.01565236, -0.01979998, -0.05413245, -0.37604701,  0.00626531,
   0.02791119,  0.30086121, -0.00214749, -0.00390029,  0.08630184,  0.11340987,
   0.07794295, -0.03393694]$$